# лабораторная работа 2

In [ ]:
%pip install PyQt5==5.15.11

## подключение библиотек

In [ ]:
from pathlib import Path

import PyQt5
from PyQt5.QtCore import QObject, QCoreApplication, QLocale, pyqtSignal, pyqtSlot
from PyQt5.QtWidgets import QApplication, QWidget, QLabel, QDoubleSpinBox, QFormLayout, QVBoxLayout

plugins_path = Path(PyQt5.__file__).resolve().parent / "Qt5" / "plugins"
QCoreApplication.setLibraryPaths([str(plugins_path)])

## обработка событий в ноутбуке

In [ ]:
%gui

## данные и пересчёт

In [ ]:
class CurrencyData(QObject):
    changed = pyqtSignal(float, float, float)

    def __init__(self):
        super().__init__()
        self.dollar_rate = 90.0
        self.euro_rate = 100.0
        self.rubles = 0.0

    @pyqtSlot(float)
    def from_rubles(self, value):
        self.rubles = value
        self.send_values()

    @pyqtSlot(float)
    def from_dollars(self, value):
        self.rubles = value * self.dollar_rate
        self.send_values()

    @pyqtSlot(float)
    def from_euros(self, value):
        self.rubles = value * self.euro_rate
        self.send_values()

    def send_values(self):
        dollars = self.rubles / self.dollar_rate
        euros = self.rubles / self.euro_rate
        self.changed.emit(self.rubles, dollars, euros)

## окно программы

In [ ]:
class CurrencyWindow(QWidget):
    def __init__(self):
        super().__init__()
        self.data = CurrencyData()
        self.setWindowTitle("лабораторная работа 2")
        self.setMinimumWidth(430)

        title = QLabel("перевод валют")
        description = QLabel("измените сумму в любом поле")
        rates = QLabel("учебный курс\n1 доллар = 90 рублей\n1 евро = 100 рублей")

        self.rubles = self.make_field(90000000)
        self.dollars = self.make_field(1000000)
        self.euros = self.make_field(900000)

        form = QFormLayout()
        form.addRow("рубли", self.rubles)
        form.addRow("доллары", self.dollars)
        form.addRow("евро", self.euros)

        layout = QVBoxLayout(self)
        layout.setContentsMargins(24, 24, 24, 24)
        layout.setSpacing(16)
        layout.addWidget(title)
        layout.addWidget(description)
        layout.addLayout(form)
        layout.addWidget(rates)

        self.rubles.valueChanged.connect(self.data.from_rubles)
        self.dollars.valueChanged.connect(self.data.from_dollars)
        self.euros.valueChanged.connect(self.data.from_euros)
        self.data.changed.connect(self.show_values)

    def make_field(self, maximum):
        field = QDoubleSpinBox()
        field.setDecimals(2)
        field.setRange(0, maximum)
        field.setSingleStep(1)
        field.setLocale(QLocale(QLocale.Russian, QLocale.Russia))
        field.setKeyboardTracking(False)
        return field

    @pyqtSlot(float, float, float)
    def show_values(self, rubles, dollars, euros):
        self.rubles.blockSignals(True)
        self.dollars.blockSignals(True)
        self.euros.blockSignals(True)

        self.rubles.setValue(rubles)
        self.dollars.setValue(dollars)
        self.euros.setValue(euros)

        self.rubles.blockSignals(False)
        self.dollars.blockSignals(False)
        self.euros.blockSignals(False)

## запуск

In [ ]:
app = QApplication.instance() or QApplication([])
app.setQuitOnLastWindowClosed(True)

previous_window = globals().get("window")
if previous_window is not None:
    previous_window.close()

window = CurrencyWindow()
window.show()
_ = app.exec_()